# 01 - FunnyBirds: dataset characterization & training validation

CBM + MCBM on **FunnyBirds**. This is dataset + training *validation*
(is the pipeline correct and trustworthy?), not the leakage/backwash
results -- those are the separate next-step notebooks. 10 figure/table
slots, publication quality.

> **How to use.** Each cell loads an artifact produced by the training / data
> scripts (see `curated/README.md`). Before those run, cells print a `[pending]`
> note instead of failing, so the notebook always executes end to end. On adroit,
> after training, every figure/table fills in and is paper-ready (saved to
> `curated/notebooks/figures/`).

In [ ]:
# Setup -- run on the cluster where CURATED_DATA and the run outputs exist.
import os, sys, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

REPO = Path.cwd()
while not (REPO / 'curated').exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
from curated.analysis import plotting, occlusion, io
plotting.set_paper_style()

DATA = Path(os.environ.get('CURATED_DATA', REPO / 'curated' / '_demo_data'))
RUNS = DATA / 'runs'

def maybe(path, loader=pd.read_parquet):
    '''Load an artifact or return None with a clear note, so the notebook',
    runs top-to-bottom before every result exists.'''
    p = Path(path)
    if not p.exists():
        print(f'[pending] {p} not found -- run the producing step on adroit.')
        return None
    return loader(p)

In [ ]:
# Concept schema comes from the OFFICIAL FunnyBirds parts.json (no hand-written module).
sys.path.insert(0, str(REPO / 'curated' / 'data' / 'funnybirds'))
import funnybirds_concepts as fbc
FB_ROOT = Path(os.environ.get('FUNNYBIRDS_ROOT', DATA / 'FunnyBirds'))
PARTS = fbc.load_parts(FB_ROOT) if (FB_ROOT / 'parts.json').exists() else None
if PARTS:
    CONCEPTS = fbc.concept_names(PARTS); SPANS = fbc.group_slices(PARTS)
    print(len(CONCEPTS), 'concepts in', len(SPANS), 'part groups:', {p: len(v) for p, v in PARTS.items()})
else:
    CONCEPTS, SPANS = None, None; print('[pending] parts.json not found at', FB_ROOT)

## 1. Dataset summary

In [ ]:
rows = []
for split in ('train','val','test'):
    recs = maybe(DATA/'funnybirds_processed'/f'{split}.pkl', lambda p: pd.read_pickle(p))
    if recs is None: continue
    A = np.array([r['attribute_label'] for r in recs])
    rows.append({'split':split, 'n_images':len(recs), 'n_classes':len({r['class_label'] for r in recs}),
                 'n_concepts':A.shape[1], 'prevalence_min':A.mean(0).min().round(3),
                 'prevalence_max':A.mean(0).max().round(3)})
summary = pd.DataFrame(rows); summary

## 2. Class-concept matrix (confirms one-hot-per-part loading)

In [ ]:
recs = maybe(DATA/'funnybirds_processed'/'train.pkl', lambda p: pd.read_pickle(p))
if recs is not None and SPANS is not None:
    df = pd.DataFrame([r['attribute_label'] for r in recs])
    df['cls'] = [r['class_label'] for r in recs]
    M = df.groupby('cls').mean()
    fig, ax = plt.subplots(figsize=(8,6))
    im = ax.imshow(M.values, aspect='auto', cmap='viridis')
    ax.set_xlabel('concept'); ax.set_ylabel('class'); ax.set_title('Class x concept (mean label)')
    for s,(a,b) in SPANS.items(): ax.axvline(b-0.5, color='w', lw=0.5)
    fig.colorbar(im, ax=ax, fraction=0.03); plotting.savefig('fb_class_concept_matrix')

## 3. CBM training curves (concept loss & task loss)

In [ ]:
# Expects a parsed-log table; produce it from the official trainer's stdout/CSV.
hist = maybe(RUNS/'funnybirds_cbm_seed1'/'history.parquet')
if hist is not None:
    fig, ax = plt.subplots()
    for col,lab in [('concept_loss','concept'),('task_loss','task')]:
        if col in hist: ax.plot(hist['epoch'], hist[col], label=lab)
    ax.set_xlabel('epoch'); ax.set_ylabel('loss'); ax.legend(); ax.set_title('CBM training')
    plotting.savefig('fb_cbm_curves')

## 4. CBM final metrics

In [ ]:
ev = maybe(RUNS/'funnybirds_cbm_seed1'/'eval_test.parquet', io.load_eval_table)
if ev is not None:
    per = ev.groupby('concept_name').apply(lambda d:(d.gt_label==d.pred_label).mean())
    img = ev.drop_duplicates('image')
    print('task acc', round((img.y_true==img.y_pred).mean(),3),
          '| mean concept acc', round(per.mean(),3))
    per.describe().to_frame('concept_acc')

## 5. MCBM training curves (task, concept, z-regularizer)

Watch the z-regularizer term and the z histogram (slot 8): the failure
mode is z collapsing to 0 (sigmoid 0.5). The +-3 target should prevent it.

In [ ]:
hist = maybe(RUNS/'funnybirds_mcbm_seed42'/'history.parquet')
if hist is not None:
    fig, ax = plt.subplots()
    for col in ('task_loss','concept_loss','z_loss'):
        if col in hist: ax.plot(hist['epoch'], hist[col], label=col)
    ax.set_xlabel('epoch'); ax.set_ylabel('loss'); ax.legend(); ax.set_title('MCBM training')
    plotting.savefig('fb_mcbm_curves')

## 6. MCBM final metrics

In [ ]:
ev_m = maybe(RUNS/'funnybirds_mcbm_seed42'/'eval_test.parquet', io.load_eval_table)
if ev_m is not None:
    per = ev_m.groupby('concept_name').apply(lambda d:(d.gt_label==d.pred_label).mean())
    img = ev_m.drop_duplicates('image')
    print('task acc', round((img.y_true==img.y_pred).mean(),3),
          '| mean concept acc', round(per.mean(),3))

## 7. CBM vs MCBM (task acc & mean concept acc)

In [ ]:
def metrics(ev):
    if ev is None: return (np.nan, np.nan)
    img = ev.drop_duplicates('image')
    return ((img.y_true==img.y_pred).mean(), (ev.gt_label==ev.pred_label).mean())
vals = {'CBM':metrics(ev), 'MCBM':metrics(ev_m)}
fig, ax = plt.subplots()
x = np.arange(2)
for i,(k,(ta,ca)) in enumerate(vals.items()):
    ax.bar(x+i*0.35, [ta,ca], 0.35, label=k, color=plotting.PALETTE[k])
ax.set_xticks(x+0.175); ax.set_xticklabels(['task acc','concept acc']); ax.legend()
ax.set_ylim(0,1); plotting.savefig('fb_cbm_vs_mcbm')

## 8. z distribution across concepts (fixed-point sanity check)

In [ ]:
fig, ax = plt.subplots()
for ev,k in [(ev,'CBM'),(ev_m,'MCBM')]:
    if ev is not None: ax.hist(ev['z'], bins=60, alpha=0.5, label=k, color=plotting.PALETTE[k], density=True)
ax.axvline(0, color='k', lw=0.8, ls='--'); ax.set_xlabel('z (pre-sigmoid)'); ax.legend()
ax.set_title('z distribution -- spike at 0 would signal the collapse bug')
plotting.savefig('fb_z_distribution')

## 9. Example images: predicted vs GT concepts

In [ ]:
# Qualitative spot check; renders a few test images with the per-group
# predicted vs GT one-hot. Fill image paths from the eval table.
if ev is not None:
    ex = ev[ev.image.isin(ev.image.drop_duplicates().head(3))]
    print(ex.pivot_table(index='image', columns='concept_name', values=['gt_label','pred_label']).head())

## 10. Intervention sanity check (acc vs #concepts intervened)

The Koh et al. headline plot and the MCBM paper's central comparison;
included pre-results because it validates training correctness.

In [ ]:
tti = maybe(RUNS/'funnybirds_tti.parquet')  # columns: model, n_intervened, task_acc
if tti is not None:
    fig, ax = plt.subplots()
    for k,g in tti.groupby('model'):
        ax.plot(g.n_intervened, g.task_acc, marker='o', label=k,
                color=plotting.PALETTE.get(k))
    ax.set_xlabel('# concepts intervened'); ax.set_ylabel('task acc'); ax.legend()
    plotting.savefig('fb_intervention')